In [16]:
from pathlib import Path

print("END-TO-END PROJECT TESTING")
print("=" * 50)

# Project root
PROJECT_ROOT = Path.cwd().parent

print("Project Root:")
print(PROJECT_ROOT.resolve())

# PROJECT FOLDERS

print("\nPROJECT FOLDERS")
print("=" * 50)

folders = [
    "data/raw",
    "data/processed",
    "models",
    "reports",
    "notebooks"
]

for folder in folders:
    path = PROJECT_ROOT / folder

    if path.exists() and path.is_dir():
        print(f"PASS: {folder}")
    else:
        print(f"FAIL: {folder}")

# KEY PROJECT FILES

print("\nKEY PROJECT FILES")
print("=" * 50)

files = [
    "data/processed/feature_engineered_dataset_final.csv",
    "models/baseline_random_forest.pkl",
    "models/final_random_forest.pkl",
    "reports/baseline_predictions.csv",
    "reports/model_evaluation_metrics.csv",
    "reports/final_model_validation_metrics.csv",
    "reports/risk_scoring.csv"
]

for file in files:
    path = PROJECT_ROOT / file

    if path.exists() and path.is_file():
        size_mb = path.stat().st_size / (1024 * 1024)

        if path.stat().st_size > 0:
            print(f"PASS: {file} | {size_mb:.2f} MB")
        else:
            print(f"FAIL: {file} | EMPTY FILE")
    else:
        print(f"FAIL: {file} | NOT FOUND")

END-TO-END PROJECT TESTING
Project Root:
C:\Users\user\Documents\Zidio_Project\FORESIGHT

PROJECT FOLDERS
PASS: data/raw
PASS: data/processed
PASS: models
PASS: reports
PASS: notebooks

KEY PROJECT FILES
PASS: data/processed/feature_engineered_dataset_final.csv | 8.55 MB
PASS: models/baseline_random_forest.pkl | 456.73 MB
PASS: models/final_random_forest.pkl | 20.91 MB
PASS: reports/baseline_predictions.csv | 7.97 MB
PASS: reports/model_evaluation_metrics.csv | 0.00 MB
PASS: reports/final_model_validation_metrics.csv | 0.00 MB
PASS: reports/risk_scoring.csv | 9.90 MB


In [17]:
# STEP 2: VALIDATE FINAL DATASET → FINAL MODEL CONNECTION

import pandas as pd
import joblib
import numpy as np

print("FINAL DATASET → FINAL MODEL CONNECTION TEST")
print("=" * 60)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "feature_engineered_dataset_final.csv"
)

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "final_random_forest.pkl"
)

# Load final dataset
test_df = pd.read_csv(DATA_PATH)

# Load final model
test_model = joblib.load(MODEL_PATH)

# Convert date columns
test_df["date"] = pd.to_datetime(test_df["date"])
test_df["launch_date"] = pd.to_datetime(test_df["launch_date"])

# Get model features
model_features = list(test_model.feature_names_in_)

# Prepare model input
X_test_connection = test_df[model_features]

missing_inputs = X_test_connection.isnull().sum().sum()

print("Dataset rows:", len(test_df))
print("Model features:", len(model_features))
print("Missing model input values:", missing_inputs)

# Prediction test
print("\nRunning model prediction test...")

connection_predictions = test_model.predict(X_test_connection)

missing_predictions = np.isnan(connection_predictions).sum()

print("Prediction rows:", len(connection_predictions))
print("Prediction missing values:", missing_predictions)

if (
    len(connection_predictions) == len(test_df)
    and missing_inputs == 0
    and missing_predictions == 0
):
    print("\nPASS: Final Dataset → Final Model connection is working.")
else:
    print("\nFAIL: Final Dataset → Final Model connection requires investigation.")

FINAL DATASET → FINAL MODEL CONNECTION TEST


Dataset rows: 73000
Model features: 15
Missing model input values: 0

Running model prediction test...
Prediction rows: 73000
Prediction missing values: 0

PASS: Final Dataset → Final Model connection is working.


In [18]:
# STEP 3: VALIDATE EXISTING RISK SCORING OUTPUT

print("\n\nRISK SCORING OUTPUT TEST")
print("=" * 60)

RISK_PATH = PROJECT_ROOT / "reports" / "risk_scoring.csv"

risk_df = pd.read_csv(RISK_PATH)

print("Rows:", len(risk_df))
print("Columns:", len(risk_df.columns))

print("\nColumns:")
print(risk_df.columns.tolist())

missing_risk_values = risk_df.isnull().sum().sum()
duplicate_risk_rows = risk_df.duplicated().sum()

print("\nTotal missing values:", missing_risk_values)
print("Duplicate rows:", duplicate_risk_rows)

print("\nFirst 5 rows:")
print(risk_df.head())

if missing_risk_values == 0 and duplicate_risk_rows == 0:
    print("\nPASS: Risk scoring output is complete.")
else:
    print("\nFAIL: Risk scoring output requires investigation.")



RISK SCORING OUTPUT TEST


C:\Users\user\AppData\Local\Temp\ipykernel_11460\3891747903.py:8: DtypeWarning: Columns (0: promo_event) have mixed types. Specify dtype option on import or set low_memory=False.
  risk_df = pd.read_csv(RISK_PATH)


Rows: 73000
Columns: 29

Columns:
['date', 'sku_id', 'units_sold', 'revenue', 'unit_price', 'promo_flag', 'category', 'subcategory', 'launch_date', 'unit_cost', 'list_price', 'week', 'month', 'season', 'is_holiday', 'promo_event', 'on_hand_units', 'on_order_units', 'lead_time_days', 'reorder_point', 'day', 'day_of_week', 'quarter', 'year', 'Predicted_Units_Sold', 'Inventory_Gap', 'Inventory_Coverage', 'Risk_Score', 'Risk_Level']

Total missing values: 261600
Duplicate rows: 0

First 5 rows:
         date  sku_id  units_sold  revenue  unit_price  promo_flag  \
0  2024-01-01  SKU001          16  3814.08      238.38           0   
1  2024-01-01  SKU002           9   943.65      104.85           0   
2  2024-01-01  SKU003          14  4658.92      332.78           0   
3  2024-01-01  SKU004          11  2825.35      256.85           0   
4  2024-01-01  SKU005          12   575.40       47.95           0   

     category subcategory launch_date  unit_cost  ...  reorder_point  day  \
0  App

In [19]:
# STEP 4: VALIDATE RISK SCORE AND RISK LEVEL

print("\n\nRISK SCORE AND RISK LEVEL VALIDATION")
print("=" * 60)

risk_columns = [
    "Predicted_Units_Sold",
    "Inventory_Gap",
    "Inventory_Coverage",
    "Risk_Score",
    "Risk_Level"
]

print("Required risk columns:")
print(risk_columns)

missing_required_columns = [
    column for column in risk_columns
    if column not in risk_df.columns
]

print("\nMissing required columns:")
print(missing_required_columns)

if not missing_required_columns:

    print("\nMissing values in risk columns:")
    print(risk_df[risk_columns].isnull().sum())

    print("\nRisk Level values:")
    print(risk_df["Risk_Level"].value_counts(dropna=False))

    print("\nRisk Score statistics:")
    print(risk_df["Risk_Score"].describe())

    print("\nInventory Coverage statistics:")
    print(risk_df["Inventory_Coverage"].describe())

    print("\nInventory Gap statistics:")
    print(risk_df["Inventory_Gap"].describe())

    # Validate Risk Score range
    invalid_risk_scores = (
        (risk_df["Risk_Score"] < 0)
        | (risk_df["Risk_Score"] > 100)
    ).sum()

    missing_risk_levels = risk_df["Risk_Level"].isnull().sum()

    print("\nInvalid Risk Scores:", invalid_risk_scores)
    print("Missing Risk Levels:", missing_risk_levels)

    if (
        invalid_risk_scores == 0
        and missing_risk_levels == 0
        and risk_df[risk_columns].isnull().sum().sum() == 0
    ):
        print("\nPASS: Risk Score and Risk Level validation successful.")
    else:
        print("\nFAIL: Risk Score or Risk Level validation requires investigation.")

else:
    print("\nFAIL: Required risk columns are missing.")

print("\n" + "=" * 60)
print("STEPS 2–4 VALIDATION COMPLETED")
print("=" * 60)



RISK SCORE AND RISK LEVEL VALIDATION
Required risk columns:
['Predicted_Units_Sold', 'Inventory_Gap', 'Inventory_Coverage', 'Risk_Score', 'Risk_Level']

Missing required columns:
[]

Missing values in risk columns:
Predicted_Units_Sold    0
Inventory_Gap           0
Inventory_Coverage      0
Risk_Score              0
Risk_Level              0
dtype: int64

Risk Level values:
Risk_Level
High      63400
Low        9549
Medium       51
Name: count, dtype: int64

Risk Score statistics:
count    73000.000000
mean        88.225710
std         30.389297
min          1.646673
25%        100.000000
50%        100.000000
75%        100.000000
max        100.000000
Name: Risk_Score, dtype: float64

Inventory Coverage statistics:
count    73000.000000
mean         1.697128
std          5.117699
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         59.728507
Name: Inventory_Coverage, dtype: float64

Inventory Gap statistics:
count    73000.000000
mean

In [22]:
# Step 5: Validate Risk Score Logic

print("RISK SCORE LOGIC VALIDATION")
print("=" * 50)

# Check Risk Score values
invalid_scores = (~np.isfinite(risk_df["Risk_Score"])).sum()

# Check Risk Score range
out_of_range_scores = (
    (risk_df["Risk_Score"] < 0) |
    (risk_df["Risk_Score"] > 100)
).sum()

# Check Inventory Coverage values
invalid_coverage = (
    ~np.isfinite(risk_df["Inventory_Coverage"])
).sum()

# Check Risk Levels
missing_risk_levels = risk_df["Risk_Level"].isnull().sum()

print("Invalid Risk Score values:", invalid_scores)
print("Risk Scores outside 0–100:", out_of_range_scores)
print("Invalid Inventory Coverage values:", invalid_coverage)
print("Missing Risk Levels:", missing_risk_levels)

print("\nRisk Score range:")
print("Minimum:", risk_df["Risk_Score"].min())
print("Maximum:", risk_df["Risk_Score"].max())

print("\nRisk Levels:")
print(risk_df["Risk_Level"].value_counts(dropna=False))

if (
    invalid_scores == 0
    and out_of_range_scores == 0
    and invalid_coverage == 0
    and missing_risk_levels == 0
):
    print("\nPASS: Risk score validation successful.")
else:
    print("\nFAIL: Risk score validation requires investigation.")

print("\nStep 5 completed.")

RISK SCORE LOGIC VALIDATION
Invalid Risk Score values: 0
Risk Scores outside 0–100: 0
Invalid Inventory Coverage values: 0
Missing Risk Levels: 0

Risk Score range:
Minimum: 1.6466731242083303
Maximum: 100.0

Risk Levels:
Risk_Level
High      63400
Low        9549
Medium       51
Name: count, dtype: int64

PASS: Risk score validation successful.

Step 5 completed.


In [23]:
# Step 6: Validate Prediction Coverage

print("PREDICTION COVERAGE VALIDATION")
print("=" * 50)

# Check final model predictions
prediction_column = "Predicted_Units_Sold"

if prediction_column in risk_df.columns:
    missing_predictions = risk_df[prediction_column].isnull().sum()
    invalid_predictions = (~np.isfinite(risk_df[prediction_column])).sum()

    print("Total rows:", len(risk_df))
    print("Missing predictions:", missing_predictions)
    print("Invalid predictions:", invalid_predictions)

    print("\nPrediction statistics:")
    print("Minimum:", risk_df[prediction_column].min())
    print("Maximum:", risk_df[prediction_column].max())
    print("Mean:", risk_df[prediction_column].mean())

    if missing_predictions == 0 and invalid_predictions == 0:
        print("\nPASS: All prediction values are valid.")
    else:
        print("\nFAIL: Prediction coverage requires investigation.")

else:
    print("FAIL: Predicted_Units_Sold column not found.")

print("\nStep 6 completed.")

PREDICTION COVERAGE VALIDATION
Total rows: 73000
Missing predictions: 0
Invalid predictions: 0

Prediction statistics:
Minimum: 2.27
Maximum: 26.52
Mean: 11.897830547945205

PASS: All prediction values are valid.

Step 6 completed.


In [24]:
# Step 7: Validate Project Output Consistency

print("PROJECT OUTPUT CONSISTENCY VALIDATION")
print("=" * 50)

# Validate row counts across important outputs
dataset_rows = len(test_df)
risk_rows = len(risk_df)
prediction_rows = len(connection_predictions)

print("Final dataset rows:", dataset_rows)
print("Risk scoring rows:", risk_rows)
print("Model prediction rows:", prediction_rows)

print("\nRow count comparison:")

if dataset_rows == risk_rows == prediction_rows:
    print("PASS: All project outputs contain the same number of rows.")
else:
    print("FAIL: Row count mismatch detected.")

# Validate required risk columns
required_risk_columns = [
    "Predicted_Units_Sold",
    "Inventory_Gap",
    "Inventory_Coverage",
    "Risk_Score",
    "Risk_Level"
]

missing_risk_columns = [
    col for col in required_risk_columns
    if col not in risk_df.columns
]

print("\nRequired risk columns:")
print(required_risk_columns)

print("\nMissing risk columns:")
print(missing_risk_columns)

if not missing_risk_columns:
    print("PASS: All required risk-scoring columns are present.")
else:
    print("FAIL: Required risk-scoring columns are missing.")

# Final Step 7 status
if (
    dataset_rows == risk_rows == prediction_rows
    and not missing_risk_columns
):
    print("\nPASS: Project output consistency validation successful.")
else:
    print("\nFAIL: Project output consistency requires investigation.")

print("\nStep 7 completed.")

PROJECT OUTPUT CONSISTENCY VALIDATION
Final dataset rows: 73000
Risk scoring rows: 73000
Model prediction rows: 73000

Row count comparison:
PASS: All project outputs contain the same number of rows.

Required risk columns:
['Predicted_Units_Sold', 'Inventory_Gap', 'Inventory_Coverage', 'Risk_Score', 'Risk_Level']

Missing risk columns:
[]
PASS: All required risk-scoring columns are present.

PASS: Project output consistency validation successful.

Step 7 completed.


In [25]:
# Step 8: Validate Final Project Data Integrity

print("FINAL PROJECT DATA INTEGRITY VALIDATION")
print("=" * 50)

# Final dataset checks
dataset_missing = test_df.isnull().sum().sum()
dataset_duplicates = test_df.duplicated().sum()

# Risk output checks
risk_missing = risk_df.isnull().sum().sum()
risk_duplicates = risk_df.duplicated().sum()

print("FINAL DATASET")
print("Rows:", len(test_df))
print("Columns:", len(test_df.columns))
print("Missing values:", dataset_missing)
print("Duplicate rows:", dataset_duplicates)

print("\nRISK SCORING OUTPUT")
print("Rows:", len(risk_df))
print("Columns:", len(risk_df.columns))
print("Missing values:", risk_missing)
print("Duplicate rows:", risk_duplicates)

# Check key identifiers
print("\nKEY IDENTIFIER CHECK")

print("Unique SKUs:", test_df["sku_id"].nunique())
print("Date range:", test_df["date"].min(), "to", test_df["date"].max())

# Final status
if (
    dataset_missing == 0
    and dataset_duplicates == 0
    and risk_missing == 0
    and risk_duplicates == 0
):
    print("\nPASS: Final project data integrity validation successful.")
else:
    print("\nFAIL: Data integrity issues detected.")

print("\nStep 8 completed.")

FINAL PROJECT DATA INTEGRITY VALIDATION
FINAL DATASET
Rows: 73000
Columns: 24
Missing values: 0
Duplicate rows: 0

RISK SCORING OUTPUT
Rows: 73000
Columns: 29
Missing values: 261600
Duplicate rows: 0

KEY IDENTIFIER CHECK
Unique SKUs: 200
Date range: 2024-01-01 00:00:00 to 2024-12-30 00:00:00

FAIL: Data integrity issues detected.

Step 8 completed.


In [26]:
# Step 8A: Identify Missing Values in Risk Scoring Output

print("RISK SCORING MISSING VALUE INVESTIGATION")
print("=" * 50)

missing_by_column = risk_df.isnull().sum()

print("Missing values by column:")
print(missing_by_column[missing_by_column > 0])

print("\nTotal missing values:", missing_by_column.sum())

print("\nColumns with missing values:")
print(missing_by_column[missing_by_column > 0].index.tolist())

RISK SCORING MISSING VALUE INVESTIGATION
Missing values by column:
promo_event       71400
on_order_units    63400
lead_time_days    63400
reorder_point     63400
dtype: int64

Total missing values: 261600

Columns with missing values:
['promo_event', 'on_order_units', 'lead_time_days', 'reorder_point']


In [27]:
# Step 8B: Inspect Risk Scoring Missing-Value Patterns

print("RISK SCORING MISSING-VALUE PATTERN")
print("=" * 50)

for column in [
    "promo_event",
    "on_order_units",
    "lead_time_days",
    "reorder_point"
]:
    print(f"\n--- {column} ---")
    print("Missing:", risk_df[column].isnull().sum())
    print("Non-missing:", risk_df[column].notnull().sum())
    print("Sample non-missing values:")
    print(risk_df[column].dropna().head(5).tolist())

print("\nRisk scoring columns:")
print(risk_df.columns.tolist())

RISK SCORING MISSING-VALUE PATTERN

--- promo_event ---
Missing: 71400
Non-missing: 1600
Sample non-missing values:
['Festival', 'Festival', 'Festival', 'Festival', 'Festival']

--- on_order_units ---
Missing: 63400
Non-missing: 9600
Sample non-missing values:
[95.0, 59.0, 86.0, 11.0, 68.0]

--- lead_time_days ---
Missing: 63400
Non-missing: 9600
Sample non-missing values:
[13.0, 5.0, 15.0, 15.0, 6.0]

--- reorder_point ---
Missing: 63400
Non-missing: 9600
Sample non-missing values:
[71.0, 29.0, 74.0, 20.0, 49.0]

Risk scoring columns:
['date', 'sku_id', 'units_sold', 'revenue', 'unit_price', 'promo_flag', 'category', 'subcategory', 'launch_date', 'unit_cost', 'list_price', 'week', 'month', 'season', 'is_holiday', 'promo_event', 'on_hand_units', 'on_order_units', 'lead_time_days', 'reorder_point', 'day', 'day_of_week', 'quarter', 'year', 'Predicted_Units_Sold', 'Inventory_Gap', 'Inventory_Coverage', 'Risk_Score', 'Risk_Level']


In [28]:
# Step 8C: Validate Critical Risk Output Fields

print("CRITICAL RISK OUTPUT VALIDATION")
print("=" * 50)

critical_risk_columns = [
    "Predicted_Units_Sold",
    "Inventory_Gap",
    "Inventory_Coverage",
    "Risk_Score",
    "Risk_Level"
]

critical_missing = risk_df[critical_risk_columns].isnull().sum()
total_critical_missing = critical_missing.sum()

print("Critical risk columns:")
print(critical_risk_columns)

print("\nMissing values in critical risk columns:")
print(critical_missing)

print("\nTotal missing critical values:", total_critical_missing)

if total_critical_missing == 0:
    print("\nPASS: All critical risk output fields are complete.")
else:
    print("\nFAIL: Critical risk output fields contain missing values.")

print("\nStep 8C completed.")

CRITICAL RISK OUTPUT VALIDATION
Critical risk columns:
['Predicted_Units_Sold', 'Inventory_Gap', 'Inventory_Coverage', 'Risk_Score', 'Risk_Level']

Missing values in critical risk columns:
Predicted_Units_Sold    0
Inventory_Gap           0
Inventory_Coverage      0
Risk_Score              0
Risk_Level              0
dtype: int64

Total missing critical values: 0

PASS: All critical risk output fields are complete.

Step 8C completed.


In [29]:
# Step 9: Validate Final Project Model and Risk Output Together

print("FINAL MODEL + RISK OUTPUT INTEGRATION TEST")
print("=" * 50)

# Model prediction count
model_prediction_count = len(connection_predictions)

# Risk output prediction count
risk_prediction_count = risk_df["Predicted_Units_Sold"].notna().sum()

# Critical risk output count
risk_score_count = risk_df["Risk_Score"].notna().sum()
risk_level_count = risk_df["Risk_Level"].notna().sum()

print("Model predictions:", model_prediction_count)
print("Risk predictions:", risk_prediction_count)
print("Risk scores:", risk_score_count)
print("Risk levels:", risk_level_count)

print("\nExpected rows:", len(test_df))

# Check consistency
if (
    model_prediction_count == len(test_df)
    and risk_prediction_count == len(test_df)
    and risk_score_count == len(test_df)
    and risk_level_count == len(test_df)
):
    print("\nPASS: Final model and risk scoring outputs are fully connected.")
else:
    print("\nFAIL: Model and risk output row counts are inconsistent.")

print("\nStep 9 completed.")

FINAL MODEL + RISK OUTPUT INTEGRATION TEST
Model predictions: 73000
Risk predictions: 73000
Risk scores: 73000
Risk levels: 73000

Expected rows: 73000

PASS: Final model and risk scoring outputs are fully connected.

Step 9 completed.


In [30]:
# Step 10: Final End-to-End Project Status

print("FINAL END-TO-END PROJECT STATUS")
print("=" * 50)

checks = {
    "Final dataset available": len(test_df) > 0,
    "Final dataset has no missing values": test_df.isnull().sum().sum() == 0,
    "Final dataset has no duplicates": test_df.duplicated().sum() == 0,
    "Final model predictions complete": len(connection_predictions) == len(test_df),
    "Risk predictions complete": risk_df["Predicted_Units_Sold"].notna().sum() == len(test_df),
    "Risk scores complete": risk_df["Risk_Score"].notna().sum() == len(test_df),
    "Risk levels complete": risk_df["Risk_Level"].notna().sum() == len(test_df),
    "Risk score values valid": (
        risk_df["Risk_Score"].between(0, 100).all()
    ),
    "Required risk columns available": all(
        col in risk_df.columns
        for col in [
            "Predicted_Units_Sold",
            "Inventory_Gap",
            "Inventory_Coverage",
            "Risk_Score",
            "Risk_Level"
        ]
    )
}

for check, result in checks.items():
    print(f"{'PASS' if result else 'FAIL'}: {check}")

print("\nPassed checks:", sum(checks.values()))
print("Total checks:", len(checks))

if all(checks.values()):
    print("\nPASS: END-TO-END PROJECT TESTING SUCCESSFUL.")
    print("The core project pipeline is working correctly.")
else:
    print("\nFAIL: Some end-to-end checks require investigation.")

print("\nStep 10 completed.")

FINAL END-TO-END PROJECT STATUS
PASS: Final dataset available
PASS: Final dataset has no missing values
PASS: Final dataset has no duplicates
PASS: Final model predictions complete
PASS: Risk predictions complete
PASS: Risk scores complete
PASS: Risk levels complete
PASS: Risk score values valid
PASS: Required risk columns available

Passed checks: 9
Total checks: 9

PASS: END-TO-END PROJECT TESTING SUCCESSFUL.
The core project pipeline is working correctly.

Step 10 completed.


In [33]:
# Step 11: Verify Required Project Files

print("REQUIRED PROJECT FILE VERIFICATION")
print("=" * 50)

required_files = [
    "data/processed/feature_engineered_dataset_final.csv",
    "models/final_random_forest.pkl",
    "reports/final_model_validation_metrics.csv",
    "reports/risk_scoring.csv",
    "reports/baseline_predictions.csv",
    "reports/model_evaluation_metrics.csv",
    "README.md",
    "requirements.txt"
]

all_files_exist = True

for file in required_files:
    path = PROJECT_ROOT / file

    exists = path.exists()
    is_file = path.is_file()
    size = path.stat().st_size if is_file else 0

    if exists and is_file and size > 0:
        print(f"PASS: {file} | {size:,} bytes")
    else:
        print(f"FAIL: {file}")
        all_files_exist = False

print("\n----------------------------------------")

if all_files_exist:
    print("PASS: All required project files are present and non-empty.")
else:
    print("FAIL: One or more required project files are missing or empty.")

print("\nStep 11 completed.")

REQUIRED PROJECT FILE VERIFICATION
PASS: data/processed/feature_engineered_dataset_final.csv | 8,962,195 bytes
PASS: models/final_random_forest.pkl | 21,927,985 bytes
PASS: reports/final_model_validation_metrics.csv | 84 bytes
PASS: reports/risk_scoring.csv | 10,380,753 bytes
PASS: reports/baseline_predictions.csv | 8,358,825 bytes
PASS: reports/model_evaluation_metrics.csv | 119 bytes
PASS: README.md | 10,221 bytes
PASS: requirements.txt | 87 bytes

----------------------------------------
PASS: All required project files are present and non-empty.

Step 11 completed.


In [32]:
# Step 11A: Verify requirements.txt Path

print("REQUIREMENTS.TXT PATH CHECK")
print("=" * 50)

requirements_path = PROJECT_ROOT / "requirements.txt"

print("PROJECT_ROOT:")
print(PROJECT_ROOT.resolve())

print("\nRequirements path:")
print(requirements_path.resolve())

print("\nExists:", requirements_path.exists())
print("Is file:", requirements_path.is_file())

if requirements_path.exists():
    print("Size:", requirements_path.stat().st_size, "bytes")

print("\nPASS: Path check completed.")

REQUIREMENTS.TXT PATH CHECK
PROJECT_ROOT:
C:\Users\user\Documents\Zidio_Project\FORESIGHT

Requirements path:
C:\Users\user\Documents\Zidio_Project\FORESIGHT\requirements.txt

Exists: True
Is file: True
Size: 87 bytes

PASS: Path check completed.


In [12]:
# Step 12: Test Dashboard Runtime

import subprocess
import sys
import time

print("DASHBOARD RUNTIME TEST")
print("=" * 50)

dashboard_path = PROJECT_ROOT / "dashboard" / "app.py"

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        str(dashboard_path),
        "--server.headless=true",
        "--server.port=8501"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(8)

if process.poll() is None:
    print("Dashboard process started successfully.")
    print("Streamlit dashboard is running on port 8501.")
    process.terminate()
else:
    output = process.stdout.read()
    print("Dashboard process stopped unexpectedly.")
    print(output)

DASHBOARD RUNTIME TEST
Dashboard process started successfully.
Streamlit dashboard is running on port 8501.


In [13]:
# Step 13: End-to-End Prediction and Risk Data Consistency

print("END-TO-END DATA CONSISTENCY TEST")
print("=" * 50)

final_df = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "feature_engineered_dataset_final.csv",
    low_memory=False
)

risk_df = pd.read_csv(
    PROJECT_ROOT / "reports" / "risk_scoring.csv",
    low_memory=False
)

print("Final dataset rows:", len(final_df))
print("Risk scoring rows:", len(risk_df))

final_keys = set(
    zip(final_df["date"].astype(str), final_df["sku_id"].astype(str))
)

risk_keys = set(
    zip(risk_df["date"].astype(str), risk_df["sku_id"].astype(str))
)

print("\nFinal dataset Date + SKU combinations:", len(final_keys))
print("Risk scoring Date + SKU combinations:", len(risk_keys))

print("Matching Date + SKU combinations:", len(final_keys & risk_keys))
print("Missing from risk scoring:", len(final_keys - risk_keys))
print("Extra in risk scoring:", len(risk_keys - final_keys))

print("\nPrediction rows in risk scoring:", risk_df["Predicted_Units_Sold"].notna().sum())
print("Risk score rows:", risk_df["Risk_Score"].notna().sum())
print("Risk level rows:", risk_df["Risk_Level"].notna().sum())

print("\nEnd-to-end data consistency test completed.")

END-TO-END DATA CONSISTENCY TEST
Final dataset rows: 73000
Risk scoring rows: 73000

Final dataset Date + SKU combinations: 73000
Risk scoring Date + SKU combinations: 73000
Matching Date + SKU combinations: 73000
Missing from risk scoring: 0
Extra in risk scoring: 0

Prediction rows in risk scoring: 73000
Risk score rows: 73000
Risk level rows: 73000

End-to-end data consistency test completed.


In [14]:
print("Final Date + SKU combinations:", len(final_keys))
print("Risk scoring Date + SKU combinations:", len(risk_keys))

print("Matching combinations:", len(final_keys & risk_keys))
print("Missing from risk scoring:", len(final_keys - risk_keys))
print("Extra in risk scoring:", len(risk_keys - final_keys))

print("\nPrediction rows:", risk_df["Predicted_Units_Sold"].notna().sum())
print("Risk score rows:", risk_df["Risk_Score"].notna().sum())
print("Risk level rows:", risk_df["Risk_Level"].notna().sum())

Final Date + SKU combinations: 73000
Risk scoring Date + SKU combinations: 73000
Matching combinations: 73000
Missing from risk scoring: 0
Extra in risk scoring: 0

Prediction rows: 73000
Risk score rows: 73000
Risk level rows: 73000


In [15]:
# Step 14: Final End-to-End Test

print("FINAL END-TO-END PROJECT TEST")
print("=" * 50)

checks = {
    "Final dataset exists": (
        PROJECT_ROOT / "data" / "processed" / "feature_engineered_dataset_final.csv"
    ).exists(),

    "Final model exists": (
        PROJECT_ROOT / "models" / "final_random_forest.pkl"
    ).exists(),

    "Final model metrics exist": (
        PROJECT_ROOT / "reports" / "final_model_validation_metrics.csv"
    ).exists(),

    "Risk scoring exists": (
        PROJECT_ROOT / "reports" / "risk_scoring.csv"
    ).exists(),

    "Dashboard exists": (
        PROJECT_ROOT / "dashboard" / "app.py"
    ).exists(),

    "Dataset rows = 73000": len(final_df) == 73000,

    "Risk rows = 73000": len(risk_df) == 73000,

    "Date + SKU coverage complete": len(final_keys - risk_keys) == 0,

    "Predictions complete": risk_df["Predicted_Units_Sold"].notna().sum() == 73000,

    "Risk scores complete": risk_df["Risk_Score"].notna().sum() == 73000,

    "Risk levels complete": risk_df["Risk_Level"].notna().sum() == 73000
}

print("\nValidation Results:")

for check, result in checks.items():
    print(f"{check}: {result}")

print("\nPassed checks:", sum(checks.values()))
print("Total checks:", len(checks))

if all(checks.values()):
    print("\nEND-TO-END TEST COMPLETED")
else:
    print("\nEND-TO-END TEST REQUIRES ATTENTION")

FINAL END-TO-END PROJECT TEST

Validation Results:
Final dataset exists: True
Final model exists: True
Final model metrics exist: True
Risk scoring exists: True
Dashboard exists: True
Dataset rows = 73000: True
Risk rows = 73000: True
Date + SKU coverage complete: True
Predictions complete: True
Risk scores complete: True
Risk levels complete: True

Passed checks: 11
Total checks: 11

END-TO-END TEST COMPLETED
